In [0]:
spark.conf.set(
  "fs.azure.account.key.smartgeardatalake01.dfs.core.windows.net",
  "Rf9Sf3PcWJtm1+vbsb+IpnT0l1N46DJgkTCCS/x+cNUNnl5xFg5+PlX+eb24+wh4wEZJos9BvETj+AStWurvtA=="
)

from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, input_file_name
from pyspark.sql.functions import col, upper, expr, year, month, to_date, sum, count, desc

In [0]:
# Read raw data (exploration)
df = spark.read.format("csv") \
    .option("header","true") \
    .load("abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv")

display(df)

OrderID,OrderDate,Region,StoreID,Product,Quantity,UnitPrice
1001,2025-03-14,East,115,Headphones,3,81.65
1002,2025-02-19,West,110,Smartwatch,3,242.5
1003,2025-03-17,West,113,Printer,2,152.59
1004,2025-01-05,West,118,Camera,4,463.4
1005,2025-01-07,West,110,Tablet,3,370.01
1006,2025-03-22,East,113,Smartphone,4,639.47
1007,2025-02-19,North,118,Drone,4,747.21
1008,2025-02-21,West,112,Printer,2,151.06
1009,2025-03-30,North,117,Smartphone,1,614.89
1010,2025-01-30,West,108,Tablet,3,415.43


In [0]:
# Read raw data (exploration)
schema = StructType([
    StructField("OrderID", IntegerType(), True),
    StructField("OrderDate", DateType(), True),
    StructField("Region", StringType(), True),
    StructField("StoreID", IntegerType(), True),
    StructField("Product", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True)
])

df_raw = spark.read.format("csv") \
    .option("header","true") \
    .schema(schema) \
    .load("abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv")

df_bronze = df_raw \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", input_file_name())

display(df_bronze)

OrderID,OrderDate,Region,StoreID,Product,Quantity,UnitPrice,_ingestion_timestamp,_source_file
1001,2025-03-14,East,115,Headphones,3,81.65,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1002,2025-02-19,West,110,Smartwatch,3,242.5,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1003,2025-03-17,West,113,Printer,2,152.59,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1004,2025-01-05,West,118,Camera,4,463.4,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1005,2025-01-07,West,110,Tablet,3,370.01,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1006,2025-03-22,East,113,Smartphone,4,639.47,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1007,2025-02-19,North,118,Drone,4,747.21,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1008,2025-02-21,West,112,Printer,2,151.06,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1009,2025-03-30,North,117,Smartphone,1,614.89,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv
1010,2025-01-30,West,108,Tablet,3,415.43,2026-03-16T21:40:46.050912Z,abfss://raw@smartgeardatalake01.dfs.core.windows.net/smartgear_sales.csv


In [0]:
# Writing Data into Bronze Container
df_bronze.write \
    .mode("overwrite") \
    .format("parquet") \
    .save("abfss://bronze@smartgeardatalake01.dfs.core.windows.net/sales/")

In [0]:
df_bronze = spark.read.format("parquet") \
.load("abfss://bronze@smartgeardatalake01.dfs.core.windows.net/sales/")

In [0]:
df_bronze.printSchema()
df_bronze.show(5)

root
 |-- OrderID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Region: string (nullable = true)
 |-- StoreID: integer (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+-------+----------+------+-------+----------+--------+---------+--------------------+--------------------+
|OrderID| OrderDate|Region|StoreID|   Product|Quantity|UnitPrice|_ingestion_timestamp|        _source_file|
+-------+----------+------+-------+----------+--------+---------+--------------------+--------------------+
|   1001|2025-03-14|  East|    115|Headphones|       3|    81.65|2026-03-16 21:40:...|abfss://raw@smart...|
|   1002|2025-02-19|  West|    110|Smartwatch|       3|    242.5|2026-03-16 21:40:...|abfss://raw@smart...|
|   1003|2025-03-17|  West|    113|   Printer|       2|   152.59|2026-03-16 2

In [0]:
# Data Cleaning
df_clean = df_bronze.filter(
    col("OrderID").isNotNull() &
    col("Quantity").isNotNull() &
    col("UnitPrice").isNotNull()
)

df_clean = df_clean.filter(
    (col("Quantity") > 0) &
    (col("UnitPrice") > 0)
)

df_clean = df_clean.withColumn(
    "Region",
    upper(col("Region"))
)

df_silver = df_clean.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)

In [0]:
#Creating Partitions

df_silver = df_silver.withColumn(
    "OrderDate", to_date(col("OrderDate"))
)

df_silver = df_silver.withColumn(
    "year", year(col("OrderDate"))
).withColumn(
    "month", month(col("OrderDate"))
)

In [0]:
# Writing to Silver Layer

df_silver.write \
.mode("overwrite") \
.partitionBy("year","month") \
.format("parquet") \
.save("abfss://silver@smartgeardatalake01.dfs.core.windows.net/sales/")

In [0]:
# Loading Data From Silver Layer
df_silver = spark.read.format("parquet") \
.load("abfss://silver@smartgeardatalake01.dfs.core.windows.net/sales/")
df_silver.show()

+-------+----------+------+-------+--------------+--------+---------+--------------------+--------------------+------------------+----+-----+
|OrderID| OrderDate|Region|StoreID|       Product|Quantity|UnitPrice|_ingestion_timestamp|        _source_file|           Revenue|year|month|
+-------+----------+------+-------+--------------+--------+---------+--------------------+--------------------+------------------+----+-----+
|   1001|2025-03-14|  EAST|    115|    Headphones|       3|    81.65|2026-03-16 21:40:...|abfss://raw@smart...|244.95000000000002|2025|    3|
|   1003|2025-03-17|  WEST|    113|       Printer|       2|   152.59|2026-03-16 21:40:...|abfss://raw@smart...|            305.18|2025|    3|
|   1006|2025-03-22|  EAST|    113|    Smartphone|       4|   639.47|2026-03-16 21:40:...|abfss://raw@smart...|           2557.88|2025|    3|
|   1009|2025-03-30| NORTH|    117|    Smartphone|       1|   614.89|2026-03-16 21:40:...|abfss://raw@smart...|            614.89|2025|    3|
|   10

In [0]:
# Region KPI Dataset
df_region_kpi = df_silver.groupBy("Region") \
.agg(
    sum("Revenue").alias("Total_Revenue"),
    count("OrderID").alias("Total_Orders"),
    sum("Quantity").alias("Total_Quantity")
)

df_region_kpi.show()

# Writing Region KPI to Gold
df_region_kpi.write \
.mode("overwrite") \
.format("parquet") \
.save("abfss://gold@smartgeardatalake01.dfs.core.windows.net/region_kpi/")

+------+-----------------+------------+--------------+
|Region|    Total_Revenue|Total_Orders|Total_Quantity|
+------+-----------------+------------+--------------+
|  WEST|         274303.2|         232|           694|
| SOUTH|        313016.64|         248|           740|
| NORTH|353551.9599999999|         280|           848|
|  EAST|        277434.47|         240|           719|
+------+-----------------+------------+--------------+



In [0]:
# Top 5 Products by Revenue
df_top_products = df_silver.groupBy("Product") \
.agg(sum("Revenue").alias("Total_Revenue")) \
.orderBy(desc("Total_Revenue")) \
.limit(5)

df_top_products.show()

# Writing Top 5 Product KPI to Gold
df_top_products.write \
.mode("overwrite") \
.format("parquet") \
.save("abfss://gold@smartgeardatalake01.dfs.core.windows.net/top_products/")

+----------+------------------+
|   Product|     Total_Revenue|
+----------+------------------+
|    Laptop|207147.61000000004|
|Smartphone|181489.55000000005|
|     Drone|         179999.84|
|    Camera|         168943.84|
|    Tablet|152735.87999999998|
+----------+------------------+



In [0]:
# Creating Store Performance Dataset
df_store_perf = df_silver.groupBy("StoreID") \
.agg(
    sum("Revenue").alias("Total_Revenue"),
    count("OrderID").alias("Total_Orders"),
    sum("Quantity").alias("Total_Quantity")
)

df_store_perf.show()

# Writing Store Performance to Gold
df_store_perf.write \
.mode("overwrite") \
.format("parquet") \
.save("abfss://gold@smartgeardatalake01.dfs.core.windows.net/store_performance/")

+-------+------------------+------------+--------------+
|StoreID|     Total_Revenue|Total_Orders|Total_Quantity|
+-------+------------------+------------+--------------+
|    108|          57774.02|          46|           141|
|    115|          69142.88|          57|           161|
|    101|          55990.12|          47|           128|
|    103|          56535.05|          51|           156|
|    111| 49561.32000000001|          43|           143|
|    120|          50465.29|          41|           120|
|    117|          74837.28|          52|           159|
|    112|          67601.26|          56|           160|
|    107|           66649.6|          51|           170|
|    114|59123.100000000006|          50|           140|
|    102| 51018.43000000001|          48|           130|
|    113|52177.270000000004|          43|           140|
|    109|          62509.11|          52|           151|
|    105|           61816.9|          49|           163|
|    110|          76987.44|   

In [0]:
# Data Quality Check
total_records = df_bronze.count()

invalid_quantity = df_bronze.filter(col("Quantity") <= 0).count()

invalid_price = df_bronze.filter(col("UnitPrice") <= 0).count()

print("Total Records:", total_records)
print("Invalid Quantity Records:", invalid_quantity)
print("Invalid Price Records:", invalid_price)

Total Records: 1000
Invalid Quantity Records: 0
Invalid Price Records: 0
